# Painting orders, window deconvolution & 3-D power spectra

JaxPM's `paint` supports four mass-assignment orders -- **NGP, CIC, TSC, PCS** --
and an optional window **deconvolution**. This notebook shows their effect on the
**3-D matter power spectrum** of an N-body run, and how the standard grid
corrections recover the true spectrum.

The corrections (applied per-mode before binning):

$$P_\mathrm{true}(k)=\frac{P_\mathrm{meas}(k)-\tfrac{1}{\bar n}\,C_\mathrm{order}(k)}{W^2(k)}$$

- **deconvolution** removes the assignment window $W(k)=\prod_i\mathrm{sinc}(k_i/2)^\mathrm{order}$,
- **dealiasing** subtracts the aliased shot noise $\tfrac{1}{\bar n}C_\mathrm{order}(k)$.

`deconvolution` is a flag on `paint(...)`; the corrections are applied at
measurement time through `power_spectrum(..., compensate_order=, shotnoise=)`
(from `jaxpm.utils`). We evolve with the cheap **symplectic** integrator
(`symplectic_ode` + diffrax `SemiImplicitEuler`) at $128^3$.

In [ ]:
import jax
import jax.numpy as jnp
import jax_cosmo as jc
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

from diffrax import (ODETerm, SaveAt, diffeqsolve, SemiImplicitEuler,
                     ConstantStepSize)

from jaxpm.pm import linear_field, lpt
from jaxpm.ode import symplectic_ode
from jaxpm.painting import paint
from jaxpm.utils import power_spectrum
from jaxpm.kernels import interpolate_power_spectrum

ORDERS = ["ngp", "cic", "tsc", "pcs"]
COLORS = {"ngp": "C3", "cic": "C0", "tsc": "C2", "pcs": "C1", "cic+deconv": "k"}

Planck18 = partial(jc.Cosmology, Omega_c=0.2607, Omega_b=0.0490, Omega_k=0.0,
                   h=0.6766, n_s=0.9665, sigma8=0.8102, w0=-1.0, wa=0.0)
cosmo = Planck18()

## 1. 3-D power spectrum ($256^3$)

A few-step symplectic N-body to $a=1$. We then **paint the final particles with each of
the four orders** and measure $P(k)$ -- raw, then deconvolved + dealiased. Raw spectra fan
out near Nyquist (window suppression + aliased shot noise); after the corrections all four
collapse onto the same curve.

In [ ]:
mesh_shape = (128, 128, 128)
box_shape = (512., 512., 512.)
t0, t1, n_steps = 0.1, 1.0, 20
nbar = np.prod(mesh_shape) / np.prod(box_shape)          # one particle per cell
knyq = np.pi * min(np.array(mesh_shape) / np.array(box_shape))

k = jnp.logspace(-3, 1, 256)
pk = jc.power.linear_matter_power(cosmo, k)
pk_fn = lambda x: interpolate_power_spectrum(x, k, pk)
ic = linear_field(mesh_shape, box_shape, pk_fn, seed=jax.random.PRNGKey(0))
dx0, p0, _ = lpt(cosmo, ic, a=t0, order=2)


def run_nbody(order, deconvolution=False):
    '''Few-step symplectic PM to a=1; returns the final displacement field.'''
    drift, kick = symplectic_ode(mesh_shape, cosmo, initial_particles='uniform',
                                 order=order, deconvolution=deconvolution)
    terms = (ODETerm(kick), ODETerm(drift))
    sol = diffeqsolve(terms, SemiImplicitEuler(), t0=t0, t1=t1,
                      dt0=(t1 - t0) / n_steps, y0=(p0, dx0), args=cosmo,
                      saveat=SaveAt(t1=True),
                      stepsize_controller=ConstantStepSize())
    return sol.ys[1][-1]                                  # final dx


# Force-painting orders we evolve (CIC field is reused for the collapse plot).
fields_dx = {o: run_nbody(o) for o in ["ngp", "cic", "pcs"]}
fields_dx["cic+deconv"] = run_nbody("cic", deconvolution=True)
print("evolved:", list(fields_dx))

In [ ]:
# Collapse plot: paint the CIC-evolved field with each order.
dx_cic = fields_dx["cic"]
raw, corrected = {}, {}
ks = None
for o in ORDERS:
    field = paint(dx_cic, initial_particles="uniform", order=o)
    ks, raw[o] = power_spectrum(field, box_shape=box_shape)
    _, corrected[o] = power_spectrum(field, box_shape=box_shape,
                                     compensate_order=o, shotnoise=(o, nbar))
ks = np.asarray(ks)
sel = ks > 0

fig, ax = plt.subplots(2, 2, figsize=(13, 9),
                       gridspec_kw=dict(height_ratios=[3, 1]), sharex="col")
for col, (data, title) in enumerate([(raw, "Raw  $P(k)$"),
                                     (corrected, "Deconvolved + dealiased")]):
    for o in ORDERS:
        ax[0, col].loglog(ks[sel], np.asarray(data[o])[sel], color=COLORS[o], label=o.upper())
        ax[1, col].semilogx(ks[sel], np.asarray(data[o])[sel] / np.asarray(data["cic"])[sel],
                            color=COLORS[o])
    ax[0, col].axvline(knyq, ls=":", color="k", lw=1)
    ax[0, col].set_title(title); ax[0, col].grid(which="both", alpha=0.3)
    ax[0, col].legend()
    ax[1, col].axvline(knyq, ls=":", color="k", lw=1)
    ax[1, col].axhspan(0.98, 1.02, color="0.6", alpha=0.3)
    ax[1, col].set_ylim(0.8, 1.3); ax[1, col].set_xlabel(r"$k$ [$h$/Mpc]")
    ax[1, col].set_ylabel("ratio to CIC"); ax[1, col].grid(which="both", alpha=0.3)
ax[0, 0].set_ylabel(r"$P(k)$  [(Mpc/$h)^3$]")
fig.suptitle("Painting order: raw spectra disagree, corrections collapse them", y=1.0)
plt.tight_layout(); plt.show()

m = sel & (ks < 0.7 * knyq)
print("max cross-order spread below 0.7 k_Nyq:")
for label, data in [("raw", raw), ("corrected", corrected)]:
    st = np.stack([np.asarray(data[o])[m] for o in ORDERS])
    print(f"  {label:9s}: {(st.max(0) / st.min(0) - 1).max():.1%}")

### Dynamical effect of the force-painting order

Above we changed only the *measurement*. The assignment order also enters the **forces**
during the sim. Here we compare the final (consistently corrected) spectra from runs whose
force loop used different orders, plus CIC with window deconvolution in the force loop.

In [ ]:
plt.figure(figsize=(7, 5))
for o in ["ngp", "cic", "pcs"]:
    field = paint(fields_dx[o], initial_particles="uniform", order=o)
    ks_, pk_ = power_spectrum(field, box_shape=box_shape,
                              compensate_order=o, shotnoise=(o, nbar))
    plt.loglog(np.asarray(ks_)[sel], np.asarray(pk_)[sel], color=COLORS[o],
               label=f"{o.upper()} forces")
field = paint(fields_dx["cic+deconv"], initial_particles="uniform", order="cic")
ks_, pk_ = power_spectrum(field, box_shape=box_shape, compensate_order="cic",
                          shotnoise=("cic", nbar))
plt.loglog(np.asarray(ks_)[sel], np.asarray(pk_)[sel], "k--",
           label="CIC forces + deconv")
plt.axvline(knyq, ls=":", color="k", lw=1)
plt.xlabel(r"$k$ [$h$/Mpc]"); plt.ylabel(r"$P(k)$  [(Mpc/$h)^3$]")
plt.title("Final $P(k)$ vs force-painting order (corrected measurement)")
plt.legend(); plt.grid(which="both", alpha=0.3); plt.tight_layout(); plt.show()